In [0]:
%pip install statsbombpy

  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-bab02ff8-7aa5-41c3-a0d2-d850fe57d634
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: attrs
    Found existing installation: attrs 24.3.0
    Not uninstalling attrs at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-bab02ff8-7aa5-41c3-a0d2-d850fe57d634
    Can't uninstall 'attrs'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from statsbombpy import sb
import pandas as pd

# Load La Liga 2020/21 matches
matches = sb.matches(competition_id=11, season_id=90)
print(f"Matches loaded: {len(matches)}")

Matches loaded: 35


/local_disk0/.ephemeral_nfs/envs/pythonEnv-bab02ff8-7aa5-41c3-a0d2-d850fe57d634/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [0]:
# Load events for all matches
all_events = []
for mid in matches['match_id'].tolist():
    all_events.append(sb.events(match_id=mid))

df_pd = pd.concat(all_events, ignore_index=True)
print(f"Total events: {len(df_pd)}")
print(df_pd.dtypes)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-bab02ff8-7aa5-41c3-a0d2-d850fe57d634/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bab02ff8-7aa5-41c3-a0d2-d850fe57d634/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bab02ff8-7aa5-41c3-a0d2-d850fe57d634/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bab02ff8-7aa5-41c3-a0d2-d850fe57d634/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bab02ff8-7aa5-41c3-a0d2-d850fe57d634/lib/python3.12/site-pack

Total events: 139030
50_50                             object
bad_behaviour_card                object
ball_receipt_outcome              object
ball_recovery_offensive           object
ball_recovery_recovery_failure    object
                                   ...  
shot_saved_to_post                object
block_save_block                  object
dribble_no_touch                  object
shot_redirect                     object
shot_follows_dribble              object
Length: 111, dtype: object


In [0]:
# Convert to Spark DataFrame
df = spark.createDataFrame(df_pd[['id','index','type','team','player',
                                   'minute','second','location',
                                   'possession_team','play_pattern']]
                           .astype(str))
df.printSchema()
df.show(5)

root
 |-- id: string (nullable = true)
 |-- index: string (nullable = true)
 |-- type: string (nullable = true)
 |-- team: string (nullable = true)
 |-- player: string (nullable = true)
 |-- minute: string (nullable = true)
 |-- second: string (nullable = true)
 |-- location: string (nullable = true)
 |-- possession_team: string (nullable = true)
 |-- play_pattern: string (nullable = true)

+--------------------+-----+-----------+----------------+------+------+------+--------+----------------+------------+
|                  id|index|       type|            team|player|minute|second|location| possession_team|play_pattern|
+--------------------+-----+-----------+----------------+------+------+------+--------+----------------+------------+
|d2bccfa7-36d1-408...|    1|Starting XI|Deportivo Alavés|   nan|     0|     0|     nan|Deportivo Alavés|Regular Play|
|2f5fae8c-fc9b-431...|    2|Starting XI|       Barcelona|   nan|     0|     0|     nan|Deportivo Alavés|Regular Play|
|e864ed70-94fa-4

In [0]:
# Event type distribution
from pyspark.sql.functions import col

df.groupBy("type").count().orderBy("count", ascending=False).show(20)

+--------------+-----+
|          type|count|
+--------------+-----+
|          Pass|40337|
| Ball Receipt*|39197|
|         Carry|32900|
|      Pressure|10832|
| Ball Recovery| 2822|
|          Duel| 1763|
|         Block| 1276|
|       Dribble| 1206|
|     Clearance| 1037|
|   Goal Keeper|  994|
|Foul Committed|  953|
|      Foul Won|  909|
|          Shot|  839|
|  Interception|  791|
| Dribbled Past|  785|
|    Miscontrol|  708|
|  Dispossessed|  672|
|  Substitution|  300|
|    Half Start|  140|
|      Half End|  140|
+--------------+-----+
only showing top 20 rows


In [0]:
# Most active players
df.groupBy("player").count().orderBy("count", ascending=False).show(10)

+--------------------+-----+
|              player|count|
+--------------------+-----+
|Lionel Andrés Mes...| 8952|
|    Jordi Alba Ramos| 8290|
|     Frenkie de Jong| 8154|
|Sergio Busquets i...| 7758|
|     Clément Lenglet| 6812|
|Pedro González López| 6012|
|Óscar Mingueza Ga...| 4622|
|   Antoine Griezmann| 4562|
|        Sergino Dest| 4038|
|     Ousmane Dembélé| 3663|
+--------------------+-----+
only showing top 10 rows


In [0]:
# Possession by team
df.groupBy("possession_team").count().orderBy("count", ascending=False).show(10)

+---------------+-----+
|possession_team|count|
+---------------+-----+
|      Barcelona|90258|
|     Villarreal| 3698|
|Atlético Madrid| 3615|
|        Sevilla| 3578|
|  Athletic Club| 3329|
|    Real Madrid| 3277|
|  Real Sociedad| 3110|
|     Celta Vigo| 3079|
|     Real Betis| 2961|
|Real Valladolid| 2896|
+---------------+-----+
only showing top 10 rows


In [0]:
# Display final summary
possession_summary = df.groupBy("possession_team").count().orderBy("count", ascending=False)
display(possession_summary)

possession_team,count
Barcelona,90258
Villarreal,3698
Atlético Madrid,3615
Sevilla,3578
Athletic Club,3329
Real Madrid,3277
Real Sociedad,3110
Celta Vigo,3079
Real Betis,2961
Real Valladolid,2896
